# Was ist data ingestion?

Um Dokumente wie PDFs, Word-Dokumente und andere Textdateien für die Verarbeitung von LLMs vorzubereiten, muss der
Text aus diesen Dokumenten extrahieren werden.
Prozess wird häufig als **Data-Ingestion** bezeichnet. Im Wesentlichen besteht der Prozess aus zwei
Schritten. Im Ersten wird der Klartext aus den Dateien extrahieren. Im zweiten Schritt wird dieser
dann in kleinere, für das LLM besser "verdauliche" Stücke geschnitten. Hier wird vom "Text-Splitting"
oder "Chunking" gesprochen. Die einzelnen Textschnipsel werden als "Chunks" bezeichnet.


In diesem Notebook wird der Prozess der Data-Ingestion aufgezeigt. Schauen Sie sich die PDF Datei (`/data/raw/Geschäftsordnung  Trust Council.pdf`) zu beginn an und überlegen Sie was potentielle Probleme
für die Verarbeitung sein könnten.

### 1. Laden von Dateien
Es gibt mehrere Bibliotheken und Tools, die verwendet werden können, um Text aus
Dokumenten zu extrahieren. Abhängig von der Art des Dokuments werden die passenden Bibliotheken
genutzt. `PyMuPDF` ist eine Bibliothek, die für das Extrahieren von Text aus PDF-Dateien verwendet
für PDFs, `docx2txt` wiederum für Word-Dokumente und `textract` für andere Textdateien verwenden. 

Aber auch Bibliotheken die speziell für die Arbeit mit LLMS entwickelt wurden bieten direkt
Integrationen für das Laden von Dokumenten. Der Vorteil in der Verwendung der Document-Loader in `langchain` besteht darin, dass die Integration in das
Ökosystem einfacher ist und der extrahierte Text als `langchain`-Dokumentklasse dargestellt und dann
in der Datenbank gespeichert werden kann.


**Aufgabe 1.1**: Laden von PDF-Dateien

PDF Dateien zählen zu den am häufigsten verwendeten Dateiformaten zum Veröffentlichen von
Dokumenten. Laden Sie den Quartalsbericht und verwenden Sie dazu die Integration von `PyMuPDF` in `langchain`.

[Hier](https://python.langchain.com/v0.2/docs/integrations/document_loaders/) ist die Dokumentation zu dem Python Modul `langchain`:

In [2]:
PDF_path = 'data/raw/Geschäftsordnung  Trust Council.pdf'

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader

In [4]:
pdf_loader = PyMuPDFLoader(PDF_path)

Die Methode `load` lädt das PDF-File in ein langchain document. Mit `pdf[0].page_content`
können wir den Inhalt der ersten Seite der PDF anzeigen. Vergleichen Sie den extrahierten Text
mit dem Originaldokument. Was fällt auf?

In [5]:
pdf = pdf_loader.load()
print(pdf[0].page_content)

Geschäftsordnung Trust Council [at] 
 
Der Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 mit der 
Stimmenmehrheit seiner Mitglieder nachfolgende Geschäftsordnung beschlossen: 
§ 1 Zweck 
Die Geschäftsordnung wurde beschlossen um den Arbeitsmodus des TC festzulegen.  
§ 2 Aufgaben des TC-Vorsitzenden 
Soweit nicht anders vereinbart, führt der/die Vorsitzende des TC die laufenden Geschäfte und 
übernimmt die Umsetzung von Beschlüssen. 
Dazu gehören beispielsweise: 
• 
die Vorbereitung von Sitzungen 
• 
die Erteilung und der Entzug des Rederechts in Trust Council Sitzungen 
• 
der anfallende Schriftwechsel mit Geschäftsleitung, juristischen Beratern sowie weiteren 
Parteien 
• 
die Vertretung des Trust Council nach außen 
Die obigen Aufgabengebiete können auch an andere Mitglieder des TC delegiert werden. 
§ 3 Vertretung des/der TC-Vorsitzenden 
1. Im Falle der Verhinderung des/der TC-Vorsitzenden nimmt der/die stellvertretende TC-
Vorsitzende seine

Die Bibliothek `PyMuPDF`, die hier im Hintergrund genutzt wird bietet die Möglichkeit den Text zu
sortieren. Dazu muss das Argument `sort=True` bei der Initialisierung des Document-Loaders als
Parameter übergeben werden.

In [6]:
pdf_loader = PyMuPDFLoader(PDF_path, sort=True)

In [7]:
pdf = pdf_loader.load()
print(pdf[0].page_content)

Geschäftsordnung Trust Council [at] 
 
Der Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 mit der 
Stimmenmehrheit seiner Mitglieder nachfolgende Geschäftsordnung beschlossen: 
§ 1 Zweck 
Die Geschäftsordnung wurde beschlossen um den Arbeitsmodus des TC festzulegen.  
§ 2 Aufgaben des TC-Vorsitzenden 
Soweit nicht anders vereinbart, führt der/die Vorsitzende des TC die laufenden Geschäfte und 
übernimmt die Umsetzung von Beschlüssen. 
Dazu gehören beispielsweise: 
• 
die Vorbereitung von Sitzungen 
• 
die Erteilung und der Entzug des Rederechts in Trust Council Sitzungen 
• 
der anfallende Schriftwechsel mit Geschäftsleitung, juristischen Beratern sowie weiteren 
Parteien 
• 
die Vertretung des Trust Council nach außen 
Die obigen Aufgabengebiete können auch an andere Mitglieder des TC delegiert werden. 
§ 3 Vertretung des/der TC-Vorsitzenden 
1. Im Falle der Verhinderung des/der TC-Vorsitzenden nimmt der/die stellvertretende TC-
Vorsitzende seine

### 2. Chunking
Der zweite Schritt der Data-Ingestion ist das sog. Text-Splitting. Der extrahierte Text wir in
kleine "Chunks" aufgeteilt. Wie im Kontext von LLMs üblich wird auch in der Größe der Chunks in
Token gerechnet. Auf dieser Seite zeigt [OpenAI](https://platform.openai.com/tokenizer) was sich
hinter einem Token verbirgt. Chunks überlappen sich in der Regel, um sicherzustellen, dass keine
Informationen  verloren gehen und das Modell auch Zusammenhänge zwischen den Chunks erkennen kann.

**Aufgabe 2.1:** Chunking

Teilen Sie den extrahierten Text mit Hilfe des `RecursiveCharacterTextSplitter` von `langchain` in  Chunks mit einer `chunk_size`  von 200 Token und einem
`chunk_overlap`von 50 Token auf. 
* Was fällt auf? 
* Welche mögliche Probleme könnte beim Verwenden der
Chunks in dieser Form auftreten?

Der [**RecursiveTextSplitter**](https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/) ist ein sehr einfacher Textsplitter und kann bei einfachen generischen
Texten verwendet werden. Er wird durch eine Liste von Zeichen parametrisiert. Es
wird versucht, diese der Reihe nach aufzuteilen, bis die Stücke klein genug sind. Die Standardliste
ist ["\n\n", "\n", " ", ""]. Dies hat den Effekt, dass versucht wird, alle Absätze (und dann Sätze,
und dann Wörter) so lange wie möglich zusammenzuhalten, da diese im Allgemeinen die am stärksten
semantisch zusammenhängenden Textstücke zu sein scheinen.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

In [10]:
chunks = text_splitter.split_documents(pdf)

for i, chunk in enumerate(chunks):
    print(f"=====Chunk {i+1}:=====\n{chunk.page_content}\n")


=====Chunk 1:=====
Geschäftsordnung Trust Council [at] 
 
Der Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 mit der 
Stimmenmehrheit seiner Mitglieder nachfolgende Geschäftsordnung beschlossen: 
§ 1 Zweck

=====Chunk 2:=====
§ 1 Zweck 
Die Geschäftsordnung wurde beschlossen um den Arbeitsmodus des TC festzulegen.  
§ 2 Aufgaben des TC-Vorsitzenden 
Soweit nicht anders vereinbart, führt der/die Vorsitzende des TC die laufenden Geschäfte und 
übernimmt die Umsetzung von Beschlüssen. 
Dazu gehören beispielsweise: 
•

=====Chunk 3:=====
Dazu gehören beispielsweise: 
• 
die Vorbereitung von Sitzungen 
• 
die Erteilung und der Entzug des Rederechts in Trust Council Sitzungen 
• 
der anfallende Schriftwechsel mit Geschäftsleitung, juristischen Beratern sowie weiteren 
Parteien 
• 
die Vertretung des Trust Council nach außen

=====Chunk 4:=====
• 
die Vertretung des Trust Council nach außen 
Die obigen Aufgabengebiete können auch an andere Mitglieder des

Das ist auffällig:
* Sätze werden abgeschnitten, Chunks können also mitten in einem Satz enden.
* Chunks Enden immer am Seitenende, da der Text an diesen Stellen getrennt wird.
* Kopf und Fußzeile wiederholen sich sehr häufig und können so in mehreren Chunks auftauchen.

Die Chunks enthalten neben dem Text weitere Metadaten. Diese ermöglichen es später genau
nachzuvollziehen aus welchen Chunk eine Information extrahiert wurde.

In [11]:
print("=====Chunk 1 Metadata:=====\n", chunks[0].metadata.keys(), "\n")
print("=====Chunk 1 Seitenzahl:=====\n", chunks[0].metadata["page"], "\n")

=====Chunk 1 Metadata:=====
 dict_keys(['source', 'file_path', 'page', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'creator', 'producer', 'creationDate', 'modDate', 'trapped']) 

=====Chunk 1 Seitenzahl:=====
 0 



**Aufgabe 2.2:** Advanced Chunking

Je verständlicher und besser die Chunks getrennt werden, desto einfacher ist es für LLMs den Inhalt
zu verarbeiten. Gerade bei etwas kleineren Modellen ist es wichtig durch besseres Splitting die
Qualität zu verbessern. Ein Beispiel für anspruchsvolleres Splitting ist der SpacyTextSplitter.
Dieser basiert auf dem Spacy NLP-Toolkit und kann Texte in Sätze und Wörter aufteilen und Satzende
und den Kontext berücksichtigen. Der SpacyTextSplitter ist jedoch langsamer als der RecursiveTextSplitter.

In [12]:
from langchain_text_splitters import SpacyTextSplitter
text_splitter = SpacyTextSplitter(chunk_size=500, chunk_overlap=100, pipeline="sentencizer")

In [13]:
spacy_chunks = text_splitter.split_documents(pdf)
for i, chunk in enumerate(spacy_chunks):
    print(f"=====Chunk {i+1}:=====\n{chunk.page_content}\n")

=====Chunk 1:=====
Geschäftsordnung Trust Council [at] 
 
Der Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 mit der 
Stimmenmehrheit seiner Mitglieder nachfolgende Geschäftsordnung beschlossen: 
§ 1 Zweck 
Die Geschäftsordnung wurde beschlossen um den Arbeitsmodus des TC festzulegen.

 
§ 2 Aufgaben des TC-Vorsitzenden 
Soweit nicht anders vereinbart, führt der/die Vorsitzende des TC die laufenden Geschäfte und 
übernimmt die Umsetzung von Beschlüssen.

=====Chunk 2:=====
Dazu gehören beispielsweise: 
• 
die Vorbereitung von Sitzungen 
• 
die Erteilung und der Entzug des Rederechts in Trust Council Sitzungen 
• 
der anfallende Schriftwechsel mit Geschäftsleitung, juristischen Beratern sowie weiteren 
Parteien 
• 
die Vertretung des Trust Council nach außen 
Die obigen Aufgabengebiete können auch an andere Mitglieder des TC delegiert werden.


§ 3 Vertretung des/der TC-Vorsitzenden 
1.

=====Chunk 3:=====
§ 3 Vertretung des/der TC-Vorsitzenden 
1.

Die Unterschiede zum RecursiveTextSplitter sind gut zu sehen. Der SpacyTextSplitter trennt die
chunks zuverlässig am Satzende.

**Aufgabe 2.3:** Seitenübergreifendes Chunking

Durch die Verarbeitung der pdf mit `langchain` gibt es die Vorteile der seitenweise Verarbeitung.
Diese ermöglicht das erfassen der Metadaten und es lässt sich genau zurückverfolgen auf welcher
Seite welcher Chunk ist. Die seitenweise Verarbeitung bringt aber auch einen Nachteil mit sich. Es
kann keinen Chunk geben, der über zwei Seiten hinweg geht. Besonders bei Aufzählungen oder Tabellen,
bei denen der Header für das Verständnis notwendig ist. Leider gibt es in `langchain` keine native
Möglichkeit den Text so zu verarbeiten. Wir umgehen das Problem, indem wir die Seiten einzeln laden
und dann den `page.content` in einen String zusammenfügen.

In [14]:
pdf_text = chr(12).join([page.page_content for page in pdf])
print(pdf_text)

Geschäftsordnung Trust Council [at] 
 
Der Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 mit der 
Stimmenmehrheit seiner Mitglieder nachfolgende Geschäftsordnung beschlossen: 
§ 1 Zweck 
Die Geschäftsordnung wurde beschlossen um den Arbeitsmodus des TC festzulegen.  
§ 2 Aufgaben des TC-Vorsitzenden 
Soweit nicht anders vereinbart, führt der/die Vorsitzende des TC die laufenden Geschäfte und 
übernimmt die Umsetzung von Beschlüssen. 
Dazu gehören beispielsweise: 
• 
die Vorbereitung von Sitzungen 
• 
die Erteilung und der Entzug des Rederechts in Trust Council Sitzungen 
• 
der anfallende Schriftwechsel mit Geschäftsleitung, juristischen Beratern sowie weiteren 
Parteien 
• 
die Vertretung des Trust Council nach außen 
Die obigen Aufgabengebiete können auch an andere Mitglieder des TC delegiert werden. 
§ 3 Vertretung des/der TC-Vorsitzenden 
1. Im Falle der Verhinderung des/der TC-Vorsitzenden nimmt der/die stellvertretende TC-
Vorsitzende seine

Da wir jetzt allerdings keine `langchain`Document Objekt mehr nutzen müssen wir auf die `split_text` Methode zurückgreifen.
Diese wird trennt strings in in Chunks man verliert aber die Metadaten. Chunks sind dann nur noch
eine Liste von strings.

In [15]:
cross_page_chunks = text_splitter.split_text(pdf_text)

In [16]:
for i, chunk in enumerate(cross_page_chunks):
    print(f"=====Chunk {i+1}:=====\n{chunk}\n")

=====Chunk 1:=====
Geschäftsordnung Trust Council [at] 
 
Der Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 mit der 
Stimmenmehrheit seiner Mitglieder nachfolgende Geschäftsordnung beschlossen: 
§ 1 Zweck 
Die Geschäftsordnung wurde beschlossen um den Arbeitsmodus des TC festzulegen.

 
§ 2 Aufgaben des TC-Vorsitzenden 
Soweit nicht anders vereinbart, führt der/die Vorsitzende des TC die laufenden Geschäfte und 
übernimmt die Umsetzung von Beschlüssen.

=====Chunk 2:=====
Dazu gehören beispielsweise: 
• 
die Vorbereitung von Sitzungen 
• 
die Erteilung und der Entzug des Rederechts in Trust Council Sitzungen 
• 
der anfallende Schriftwechsel mit Geschäftsleitung, juristischen Beratern sowie weiteren 
Parteien 
• 
die Vertretung des Trust Council nach außen 
Die obigen Aufgabengebiete können auch an andere Mitglieder des TC delegiert werden.


§ 3 Vertretung des/der TC-Vorsitzenden 
1.

=====Chunk 3:=====
§ 3 Vertretung des/der TC-Vorsitzenden 
1.